# SHAARP.py Interactive Session

This notebook is the Jupyter-first interactive entry point for the staged Python port. The result panel reports validation status and physical-convention metadata with each run.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import shaarp
shaarp.__all__[:5], repo_root

(['MathematicaComparisonResult',
  'PhysicsConventions',
  'SHAARPResult',
  'ValidationStatus',
  'CrystalOrientation'],
 WindowsPath('shaarp_py'))

## Launch Controls

In [2]:
session = shaarp.make_interactive_session()
session

## Direct SI Compatibility Example

In [3]:
material = shaarp.default_interactive_material()
si_result = shaarp.run_si_numeric(
    material,
    {
        "workflow": "shaarp_si_compat",
        "normal_incidence_2omega_branch_policy": "shaarp_reference_like",
    },
)
si_result.kind, si_result.validation.status, si_result.numeric["reflected_er2w"]

('si_numeric_shaarp_si_compat',
 'mathematica_stage_validated_for_exported_36_case_si_reflected_er2w',
 array([-0.02172315+0.0120553j ,  0.00972256-0.02390209j,
         0.02172315-0.0120553j ]))

## Direct Analytical Scaffold Example

In [4]:
si_symbolic = shaarp.run_si_full_analytical(
    None,
    {"workflow": "isotropic_symbolic_scaffold", "simplify": False},
)
ml_symbolic = shaarp.run_ml_partial_analytical(
    {"point_group": "3m"},
    {"workflow": "pnl_symbolic_scaffold", "simplify": True},
)

(
    si_symbolic.kind,
    si_symbolic.validation.status,
    list(si_symbolic.stages),
    ml_symbolic.kind,
    ml_symbolic.validation.status,
    list(ml_symbolic.stages),
)

('si_full_analytical_isotropic_scaffold',
 'si_isotropic_full_analytical_published_validated__general_facade_not_full_mathematica_validated',
 ['workflow',
  'point_group',
  'linear_coefficients',
  'shg_boundary_coefficients',
  'boundary_residual',
  'source_polarizations',
  'symbol_metadata'],
 'ml_partial_analytical_pnl_scaffold',
 'python_symbolic_pnl_scaffold_not_full_mathematica_validated',
 ['workflow', 'point_group', 'd_voigt', 'partial_sources', 'symbol_metadata'])

## Direct Fresnel Example

In [5]:
system = shaarp.default_interactive_system()
result = shaarp.run_fresnel_sweep(system, angle_grid=[0, 10, 20])
result.kind, result.validation.status, result.numeric["theta_deg"], result.numeric["rp"]

('fresnel_sweep',
 'staged_python_not_fully_mathematica_validated',
 array([ 0., 10., 20.]),
 array([0.05313748, 0.05077017, 0.04365455]))

## Merged SHAARP.si + SHAARP.ml GUI (the faithful two-in-one panel)

`make_shaarp_gui()` reproduces BOTH original Mathematica GUIs as one panel with a top-level Tab to
navigate between **SHAARP.si (single interface)** and **SHAARP.ml (multilayer)** — every
Functionality mode drives the validated Python solver facades.

Per tab you get: the Functionality dropdown (SHG Simulation / analytical modes; .ml adds Maker
Fringes + Fresnel Coefficients), the point group (the free d-components rebuild automatically with
the symmetry constraints applied), refractive-index entry (ordinary + extraordinary), crystal
orientation (z-cut, or the faithful SHAARP `hklConvert` Miller mode with lattice constants +
surface hkl + in-plane uvw), the .ml layer fields (substrate indices, film thickness, wavelength),
the Assumptions panel (Full/FMR, JK, HH), case-study system presets (including the live-Mathematica
validated Quartz + Au docs configuration), material preset save/recall slots, data export (JSON),
and — after an analytical run — the **closed-form polarimetry expression** in a copyable text area
(also exported as `.txt`). Each Run renders the 2D and 3D sample schematics plus the iconic output
plot for the chosen mode.

In [6]:
gui = shaarp.make_shaarp_gui()
gui

### Scripting the same logic headlessly

The per-tab compute logic is pure (no widgets), so everything the GUI does is scriptable:

In [7]:
# the .ml Maker-fringe sweep the GUI's Run performs, headless:
ml_result = shaarp.compute_ml_gui_result(
    "Maker Fringes", theta_min_deg=0.0, theta_max_deg=45.0, theta_step_deg=5.0,
    assumption="JK", system_preset="Quartz + Au (Fig 4, 800 nm)",
)
print(sorted(ml_result.numeric))

['parallel_amplitude', 'parallel_intensity', 'perpendicular_amplitude', 'perpendicular_intensity', 'theta_deg']


In [8]:
# the SI full-analytical closed form (symbolic polarimetry) + its copyable text:
si_analytical = shaarp.compute_si_gui_result("Full Analytical", point_group="-43m", theta_deg=45.0)
print(shaarp.analytical_expression_text(si_analytical)[:400])

# si_full_analytical_polarimetry (point group -43m)

# symbols: input_polarization = phi, analyzer = psi

reflected_s_2omega =
0.0114276872450903*d14*cos(phi)**2

reflected_p_2omega =
-0.000410675962061551*d14*sin(2*phi)

analyzed_intensity =
Abs(0.000410675962061551*d14*sin(2*phi)*cos(psi) - 0.0114276872450903*d14*sin(psi)*cos(phi)**2)**2
